# فصل ۱۲: آشنایی با پایگاه داده‌های برداری (Vector Databases)
## بخش ۱: چرا دیتابیس‌های سنتی کافی نیستند؟

### 🎯 اهداف این فصل:
1. درک تفاوت بنیادین **جستجوی معنایی (Semantic Search)** با **جستجوی کلمات کلیدی (Keyword Match)**.
2. آشنایی با آناتومی وکتور دیتابیس‌ها و الگوریتم‌های جستجوی تقریبی (**ANN** مانند HNSW).
3. تسلط بر متریک‌های ریاضی شباهت (فاصله کسینوسی، اقلیدسی $L_2$ و ضرب داخلی Dot Product).
4. راه‌اندازی و کار عملی با دیتابیس **ChromaDB** در پایتون و LangChain.
5. مدیریت چرخه حیات داده‌ها (**CRUD**) و پیاده‌سازی فیلترینگ ترکیبی (**Metadata + Semantic**).

### ۱. توهم جستجو در SQL و شکست کلمات کلیدی (Keyword Search)

در دیتابیس‌های رابطه‌ای سنتی (مانند PostgreSQL یا MySQL) جستجو بر پایه تطبیق کاراکتر به کاراکتر کلمات انجام می‌شود:
```sql
SELECT * FROM books WHERE description LIKE '%داستانهای ترسناک فضایی%';
```
اگر کتابی با عنوان **«ماجرای وحشتناک در کهکشان»** در دیتابیس وجود داشته باشد، SQL هرگز آن را پیدا نمی‌کند؛ زیرا کدهای اسکی کلمات «ترسناک» و «وحشتناک» یا «فضا» و «کهکشان» کاملاً با هم متفاوتند.

- **دیتابیس سنتی:** کدام ردیف‌ها *دقیقاً* حاوی این کلمات هستند؟ (Lexical Match)
- **دیتابیس برداری:** کدام بردارها در فضای ریاضی، *نزدیک‌ترین همسایه* به بردار سوال من هستند؟ (Nearest Neighbor Search)

### ۲. آزمایش عملی: چرا نمی‌توان بردارها را با حلقه ساده (Brute-Force) در SQL جستجو کرد؟

اگر بخواهیم بردارها را به صورت ستون آرایه‌ای در SQL ذخیره کنیم و فاصله سوال را با تک‌تک رکوردهای دیتابیس محاسبه کنیم (روش Brute-Force)، با **نفرین ابعاد (Curse of Dimensionality)** و **پیچیدگی خطی $O(N)$** مواجه می‌شویم.

در سلول زیر زمان لازم برای ضرب داخلی ۱ کوئری در بین ۱۰۰ هزار بردار ۱۵۳۶ بعدی (معادل خروجی مدل‌های OpenAI و ابعاد بزرگ) را اندازه می‌گیریم:

In [1]:
import numpy as np
import time

# شبیه‌سازی ۱۰۰ هزار بردار با ۱۵۳۶ بُعد (برای تست پرفورمنس)
# نکته: ۱ میلیون بردار حدود ۶ گیگابایت از رم را اشغال می‌کند؛ در اینجا ۱۰۰ هزار رکورد را تست می‌کنیم
database_size = 100_000
dimensions = 1536

print(f"⏳ در حال تولید دیتابیس فرضی شامل {database_size:,} بردار با {dimensions} بعد...")
database = np.random.rand(database_size, dimensions).astype('float32')
query_vector = np.random.rand(1, dimensions).astype('float32')

# محاسبه شباهت با تک‌تک رکوردها (Brute-Force)
start_time = time.time()
similarities = np.dot(database, query_vector.T)
end_time = time.time()

elapsed = end_time - start_time
print(f"✅ زمان جستجو برای {database_size:,} رکورد: {elapsed:.4f} ثانیه")
print(f"📊 تخمین زمان برای ۱ میلیون رکورد: {elapsed * 10:.2f} ثانیه")
print(f"📊 تخمین زمان برای ۱۰ میلیون رکورد: {elapsed * 100:.2f} ثانیه")

⏳ در حال تولید دیتابیس فرضی شامل 100,000 بردار با 1536 بعد...
✅ زمان جستجو برای 100,000 رکورد: 0.0588 ثانیه
📊 تخمین زمان برای ۱ میلیون رکورد: 0.59 ثانیه
📊 تخمین زمان برای ۱۰ میلیون رکورد: 5.88 ثانیه


### ۳. تحلیل نتایج و چرایی شکست روش‌های سنتی:

1. **تاخیر خطی $O(N)$:** اگر بررسی ۱۰۰ هزار رکورد ۰.۰۵ ثانیه طول بکشد، برای ۱۰ میلیون رکورد بیش از ۵ ثانیه طول می‌کشد که برای سیستم‌های زنده (مثل چت‌بات‌ها و RAG) غیرقابل قبول است.
2. **فشار هم‌زمان (Concurrency):** این محاسبه ۱۰۰٪ ظرفیت یک هسته CPU را مصرف می‌کند. اگر ۱۰۰ کاربر هم‌زمان جستجو کنند، سرور قفل می‌شود.
3. **عدم امکان ساخت ایندکس‌های درختی سنتی (B-Tree):**
   - در ۱ بعد: با یک `ORDER BY` و ساختار B-Tree در زمان $O(\log N)$ سریعاً داده پیدا می‌شود.
   - در ۱۵۳۶ بعد: داده‌ها به شدت پراکنده (Sparse) هستند و نمی‌توان هم‌زمان روی ۱۵۳۶ ستون `ORDER BY` اعمال کرد!

### ۴. جدول مقایسه دیتابیس سنتی در برابر دیتابیس برداری

| ویژگی | دیتابیس سنتی (SQL / NoSQL) | دیتابیس برداری (Vector DB) |
| :--- | :--- | :--- |
| **نوع داده اصلی** | متن، عدد، تاریخ (Exact Values) | بردار عددی اعشاری (Dense Float Arrays) |
| **روش جستجو** | تطبیق کلمات کلیدی (Keyword Match) | شباهت معنایی (Semantic Similarity) |
| **سوال کاربر** | «متنی که دقیقاً حاوی کلمه X باشد» | «متنی که مفهوم و پیامش نزدیک به X باشد» |
| **چالش اصلی** | مدیریت تراکنش‌ها (ACID) و Joinها | جستجوی سریع در ابعاد بسیار بالا (ANN) |
| **نمونه خطا/شکست** | جستجوی «موبایل» و نیافتن «تلفن همراه» | فدا کردن درصد ناچیزی از دقت ریاضی به نفع سرعت بالا |

### ۵. راه‌حل انقلابی: جستجوی تقریبی نزدیک‌ترین همسایه (ANN)

در دیتابیس‌های برداری (مثل ChromaDB، Pinecone، Qdrant و Milvus)، به جای بررسی تک‌تک بردارها، از الگوریتم‌های هوشمندانه‌ای مانند **HNSW (Hierarchical Navigable Small World)** استفاده می‌شود.

در این روش:
- بردارها در گراف‌ها یا خوشه‌های چندلایه سازماندهی می‌شوند.
- به جای جستجوی ۱۰۰٪ داده‌ها، تنها چند گره مرتبط در چند میلی‌ثانیه پیمایش می‌شوند.
- **قاعده کلی وکتور دیتابیس‌ها:** ما دقت ۱۰۰٪ ریاضی را فدای سرعت می‌کنیم و به دنبال «به اندازه کافی دقیق‌ترین پاسخ در کمترین زمان ممکن» هستیم!